# Spark Fraud Detection Inference

This notebook performs inference using a trained Logistic Regression model to score new data for fraud detection. The workflow includes:

1. Data loading from ClickHouse
2. Feature preprocessing and scaling
3. Model inference
4. Results evaluation and saving

## Configuration

First, let's set up the necessary imports and configuration.

In [1]:
# Import necessary libraries
from pyspark.sql import SparkSession
from pyspark.ml import PipelineModel
from pyspark.ml.feature import VectorAssembler, StandardScaler, StringIndexer, OneHotEncoder
from pyspark.ml.classification import LogisticRegressionModel
from pyspark.ml import Pipeline
from pyspark.sql.functions import col, when
import pyspark.sql.functions as F
from datetime import datetime
import logging
import os

print("✅ Libraries imported successfully")

✅ Libraries imported successfully


In [2]:
# Configure logging
log_filename = f"spark_inference_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"
log_path = os.path.join(os.getcwd(), log_filename)

logger = logging.getLogger('spark_inference')
logger.setLevel(logging.DEBUG)

# Clear any existing handlers
for handler in logger.handlers[:]:
    logger.removeHandler(handler)

formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')

file_handler = logging.FileHandler(log_path)
file_handler.setLevel(logging.DEBUG)
file_handler.setFormatter(formatter)

console_handler = logging.StreamHandler()
console_handler.setLevel(logging.INFO)
console_handler.setFormatter(formatter)

logger.addHandler(file_handler)
logger.addHandler(console_handler)

logger.info("✅ Logging configured")
logger.info(f"📝 Logging to file: {log_path}")

2025-11-05 15:55:18,391 - spark_inference - INFO - ✅ Logging configured
2025-11-05 15:55:18,395 - spark_inference - INFO - 📝 Logging to file: /root/research-dir/dev/jazzcash-fraud-detection/scripts/spark_inference_20251105_155518.log


In [3]:
# Configuration settings
jar_files = [
    "/root/research-dir/dev/jazzcash-fraud-detection/utils/clickhouse-jdbc-0.9.2-all-dependencies.jar"
]

CLICKHOUSE_CONFIG = {
    'host': 'localhost',
    'port': 9000,
    'database': 'public',
    'user': 'default',
    'password': 'DfsTeChB1'
}

url = f"jdbc:ch://{CLICKHOUSE_CONFIG['host']}:8123/{CLICKHOUSE_CONFIG['database']}"
user = CLICKHOUSE_CONFIG['user']
password = CLICKHOUSE_CONFIG['password']
driver = "com.clickhouse.jdbc.ClickHouseDriver"

MODEL_PATH = "/root/research-dir/dev/jazzcash-fraud-detection/scripts/spark_lr_model_june"

logger.info(f"🔧 Model path: {MODEL_PATH}")
print("✅ Configuration set")

2025-11-05 15:55:18,402 - spark_inference - INFO - 🔧 Model path: /root/research-dir/dev/jazzcash-fraud-detection/scripts/spark_lr_model_june


✅ Configuration set


## Spark Session Initialization

Initialize Spark session with appropriate configurations for the cluster.

In [4]:
# Initialize Spark Session
logger.info("🚀 Initializing Spark Session for inference...")

try:
    spark.stop()
    logger.info("🔄 Stopped existing Spark session")
except:
    pass

spark = SparkSession.builder \
    .appName("fraud_inference") \
    .master("spark://dfs-ai-app2:7077") \
    .config("spark.jars", ",".join(jar_files)) \
    .config("spark.executor.memory", "100g") \
    .config("spark.executor.memoryOverhead", "5g") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.cores", "32") \
    .config("spark.executor.instances", "2") \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.default.parallelism", "96") \
    .getOrCreate()

logger.info("✅ Spark Session initialized")
print(f"Spark UI available at: {spark.sparkContext.uiWebUrl}")

2025-11-05 15:55:18,456 - spark_inference - INFO - 🚀 Initializing Spark Session for inference...
25/11/05 15:55:19 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
2025-11-05 15:55:22,611 - spark_inference - INFO - ✅ Spark Session initialized


Spark UI available at: http://dfs-ai-app2:4040


## Model Loading

Load the pre-trained Logistic Regression model.

In [5]:
# Load the trained Logistic Regression model
logger.info(f"📦 Loading trained model from: {MODEL_PATH}")

try:
    lr_model = LogisticRegressionModel.load(MODEL_PATH)
    logger.info("✅ Model loaded successfully")
except Exception as e:
    logger.error(f"❌ Error loading model: {str(e)}")
    raise

logger.info(f"   • Model intercept: {lr_model.intercept:.6f}")
logger.info(f"   • Number of features: {len(lr_model.coefficients)}")

print(f"Model intercept: {lr_model.intercept:.6f}")
print(f"Number of features: {len(lr_model.coefficients)}")

2025-11-05 15:55:22,621 - spark_inference - INFO - 📦 Loading trained model from: /root/research-dir/dev/jazzcash-fraud-detection/scripts/spark_lr_model_june
2025-11-05 15:55:26,219 - spark_inference - INFO - ✅ Model loaded successfully  
2025-11-05 15:55:26,221 - spark_inference - INFO -    • Model intercept: -10.376682
2025-11-05 15:55:26,234 - spark_inference - INFO -    • Number of features: 290


Model intercept: -10.376682
Number of features: 290


## Data Loading

Load inference data from ClickHouse database with the selected features.

In [6]:
# Define selected features and query parameters
start_date = '2025-07-01'
end_date = '2025-07-02'
num_partitions = 2

selected_cols = [
    'cutoff_date',
    'fraud_flag',
    'trans_id',
    'ac_from',
    'ac_to',
    'trx_channel',
    'trx_type',
    'start_balance',
    'trx_amt',
    'mbar_registered_channel',
    'mbar_a_c_status',
    'mbar_a_c_level',
    'mbar_account_type_name',
    'hour_of_day',
    'day_of_week',
    'is_weekend',
    'is_night',
    'is_business_hours',
    'is_unusual_hour',
    'night_weekend_combo',
    'start_balance_log',
    'txn_txns_3d',
    'txn_total_amount_3d',
    'txn_avg_amount_3d',
    'txn_max_amount_3d',
    'txn_min_amount_3d',
    'txn_unique_recipients_3d',
    'txn_unique_channels_3d',
    'txn_unique_types_3d',
    'txn_is_high_activity_3d',
    'txn_multi_channel_recent',
    'txn_amount_deviation_from_avg',
    'txn_night_txns_3d',
    'txn_weekend_txns_3d',
    'channel_new_jc_app',
    'channel_ussd',
    'channel_ussd_api',
    'channel_payment_gateway',
    'channel_mobile_app',
    'type_transfer_c2c',
    'type_transfer_c2b',
    'type_bill_payment',
    'type_mobile_load',
    'user_total_txns_3d',
    'user_total_amount_3d',
    'user_avg_amount_3d',
    'user_median_amount_3d',
    'user_max_amount_3d',
    'user_min_amount_3d',
    'user_unique_recipients_3d',
    'user_unique_channels_3d',
    'user_unique_types_3d',
    'user_total_txns_7d',
    'user_total_amount_7d',
    'user_avg_amount_7d',
    'user_median_amount_7d',
    'user_max_amount_7d',
    'user_min_amount_7d',
    'user_unique_recipients_7d',
    'user_unique_channels_7d',
    'user_unique_types_7d',
    'user_most_used_channel_7d',
    'user_last_used_channel',
    'user_channel_diversity_score_7d',
    'user_most_used_type_7d',
    'user_last_used_type',
    'user_type_diversity_score_7d',
    'user_night_txns_7d',
    'user_weekend_txns_7d',
    'user_peak_hour_txns_7d',
    'user_off_peak_hour_txns_7d',
    'user_avg_start_balance_7d',
    'user_avg_end_balance_7d',
    'user_min_balance_7d',
    'user_max_balance_7d',
    'user_balance_volatility_7d',
    'user_avg_amount_per_recipient_7d',
    'user_max_amount_to_single_recipient_7d',
    'user_recipient_concentration_ratio_7d',
    'user_avg_time_between_txns_7d',
    'user_txn_frequency_score_7d',
    'user_days_since_last_txn'
]

print(f"Selected {len(selected_cols)} features for inference")
print(f"Date range: {start_date} to {end_date}")

Selected 82 features for inference
Date range: 2025-07-01 to 2025-07-02


In [7]:
# Load inference data from ClickHouse
logger.info("📊 Loading inference data from ClickHouse...")

query = f"""
SELECT {', '.join(selected_cols)}
FROM public.stixor_fraud_features_distributed
WHERE cutoff_date BETWEEN '{start_date}' AND '{end_date}'
AND ac_to GLOBAL IN (
    SELECT b.a_c_reference
    FROM public.fraud_distributed fraud
    GLOBAL INNER JOIN public.stixor_mbar_v_distributed b
    ON fraud.fraud_msisdn = b.a_c_reference
    WHERE b.account_type_name = 'Customer Account'
    LIMIT 10
)
"""

# query = f"""
# SELECT *
# FROM public.stixor_fraud_features_distributed
# LIMIT 1000
# """

subquery = f"""
(
    {query}
) AS inference_data
"""

try:
    # Optimized JDBC options for stability
    df_inference = (spark.read
        .format('jdbc')
        .option('driver', driver)
        .option('url', url)
        .option('user', user)
        .option('password', password)
        .option('dbtable', subquery)
        .option('fetchsize', '50000')  # Reduced batch size for stability
        .option("partitionColumn", "cutoff_date")
        .option('lowerBound', start_date)
        .option('upperBound', end_date)
        .option('numPartitions', str(num_partitions))  # Reduced partitions for stability
        .load())

    logger.info("📦 Materializing and caching inference DataFrame...")
    
    # Force materialization with persist for better memory management
    df_inference.persist()
except Exception as e:
    logger.error("❌ ERROR LOADING INFERENCE DATA!")
    logger.error(f"Error: {str(e)}")
    logger.info("💡 Try reducing date range or number of partitions if connection issues persist")
    raise

2025-11-05 15:55:26,253 - spark_inference - INFO - 📊 Loading inference data from ClickHouse...
25/11/05 15:55:36 WARN JDBCRelation: The number of partitions is reduced because the specified number of partitions is less than the difference between upper bound and lower bound. Updated number of partitions: 1; Input number of partitions: 2; Lower bound: '2025-07-01'; Upper bound: '2025-07-02'.
2025-11-05 15:55:36,377 - spark_inference - INFO - 📦 Materializing and caching inference DataFrame...
25/11/05 15:55:36 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


In [8]:
# Display basic information about the loaded data
print("Data Schema:")
df_inference.printSchema()

print("\nFirst 5 rows:")
df_inference.show(5, truncate=False)

Data Schema:
root
 |-- cutoff_date: date (nullable = true)
 |-- fraud_flag: integer (nullable = true)
 |-- trans_id: string (nullable = true)
 |-- ac_from: string (nullable = true)
 |-- ac_to: string (nullable = true)
 |-- trx_channel: string (nullable = true)
 |-- trx_type: string (nullable = true)
 |-- start_balance: double (nullable = true)
 |-- trx_amt: double (nullable = true)
 |-- mbar_registered_channel: string (nullable = true)
 |-- mbar_a_c_status: string (nullable = true)
 |-- mbar_a_c_level: string (nullable = true)
 |-- mbar_account_type_name: string (nullable = true)
 |-- hour_of_day: integer (nullable = true)
 |-- day_of_week: integer (nullable = true)
 |-- is_weekend: integer (nullable = true)
 |-- is_night: integer (nullable = true)
 |-- is_business_hours: integer (nullable = true)
 |-- is_unusual_hour: integer (nullable = true)
 |-- night_weekend_combo: integer (nullable = true)
 |-- start_balance_log: double (nullable = true)
 |-- txn_txns_3d: decimal(20,0) (nullable 

+-----------+----------+--------+-------+-----+-----------+--------+-------------+-------+-----------------------+---------------+--------------+----------------------+-----------+-----------+----------+--------+-----------------+---------------+-------------------+-----------------+-----------+-------------------+-----------------+-----------------+-----------------+------------------------+----------------------+-------------------+-----------------------+------------------------+-----------------------------+-----------------+-------------------+------------------+------------+----------------+-----------------------+------------------+-----------------+-----------------+-----------------+----------------+------------------+--------------------+------------------+---------------------+------------------+------------------+-------------------------+-----------------------+--------------------+------------------+--------------------+------------------+---------------------+-----------

## Feature Preprocessing (Exact Training Match)

Apply the **exact same preprocessing steps** as used during training to ensure feature compatibility.

### Key Principles Applied:

1. **Exact Code Replication**: 
   - Using identical preprocessing code from `spark_training.py`
   - Same excluded columns list
   - Same feature engineering pipeline
   - Same scaling parameters

2. **Feature Compatibility**:
   - Target: 290 features (matching trained model)
   - Same string encoding approach
   - Same feature order and naming

3. **Data Consistency**:
   - Same null handling (`UNKNOWN` for missing strings)
   - Same StandardScaler settings (`withStd=True`, `withMean=True`)
   - Same VectorAssembler configuration

4. **Added Inference Identifiers**:
   - Keeps `trans_id`, `ac_from`, `ac_to` for result tracking
   - These are excluded from features but retained for output

In [ ]:
# Feature preprocessing - EXACTLY MATCHING TRAINING SCRIPT
TARGET_COLUMN='fraud_flag'

# # Check target variable distribution
# logger.info(f"? Target Variable ({TARGET_COLUMN}) Distribution:")
# df_inference.groupBy(TARGET_COLUMN).count().show()

# Get feature columns (exclude non-predictive columns from fraud analysis)
excluded_columns = [
    TARGET_COLUMN,  # Target variable
    'processed_date',  # Time identifier
    'ac_from',  # Account identifiers
    'ac_to', 
    'msisdn_from',
    'msisdn_to',
    'transaction_uuid',  # Transaction identifiers
    'trans_id'  # Additional identifier in inference data
]

# Get all numerical feature columns based on fraud profiling analysis
all_feature_cols = [col_name for col_name in df_inference.columns if col_name not in excluded_columns]

# Identify string (categorical) columns in all_feature_cols
string_cols = []
for field in df_inference.schema.fields:
    if field.name in all_feature_cols and field.dataType.typeName() == 'string':
        string_cols.append(field.name)

logger.info(f"🔤 String (categorical) features to encode: {string_cols}")

print(f"✅ Feature analysis completed")
print(f"   • Total columns: {len(df_inference.columns)}")
print(f"   • Excluded columns: {len(excluded_columns)}")
print(f"   • Feature columns: {len(all_feature_cols)}")
print(f"   • String columns: {len(string_cols)}")

In [ ]:
# Replace empty strings in string columns with 'UNKNOWN'
from pyspark.sql.functions import when
for col_name in string_cols:
    df_inference = df_inference.withColumn(col_name, when((F.col(col_name) == "") | F.col(col_name).isNull(), "UNKNOWN").otherwise(F.col(col_name)))

print("✅ Replaced empty strings with 'UNKNOWN'")

2025-11-05 11:47:18,102 - spark_inference - INFO - 🧹 Cleaning string columns...


✅ Replaced empty strings with 'UNKNOWN' and checkpointed data


In [ ]:
df_inference.columns

In [ ]:
# Index and encode string columns - EXACTLY MATCHING TRAINING
indexers = [
    StringIndexer(inputCol=col, outputCol=f"{col}_idx", handleInvalid="keep") for col in string_cols
]
encoders = [
    OneHotEncoder(inputCol=f"{col}_idx", outputCol=f"{col}_ohe", handleInvalid="keep") for col in string_cols
]

# Apply indexers and encoders sequentially
from pyspark.ml import Pipeline

cat_pipeline = Pipeline(stages=indexers + encoders)
df_cat = cat_pipeline.fit(df_inference).transform(df_inference)

# Replace original string columns in all_feature_cols with their OHE columns
modeling_features = [
    f"{col}_ohe" if col in string_cols else col for col in all_feature_cols
]

# Remove any columns that are not present in df_cat (e.g., if OHE dropped some columns)
modeling_features = [col for col in modeling_features if col in df_cat.columns]

logger.info(f"🧮 Final modeling features (after encoding): {len(modeling_features)} features")

# Use df_cat as the cleaned DataFrame for further processing
df_clean = df_cat

print("✅ String indexing and one-hot encoding completed")

2025-11-05 11:47:52,401 - spark_inference - INFO - 🔤 Building optimized categorical encoding pipeline...
2025-11-05 11:47:52,403 - spark_inference - INFO - Processing 10 string columns: ['trx_channel', 'trx_type', 'mbar_registered_channel', 'mbar_a_c_status', 'mbar_a_c_level', 'mbar_account_type_name', 'user_most_used_channel_7d', 'user_last_used_channel', 'user_most_used_type_7d', 'user_last_used_type']
2025-11-05 11:47:52,403 - spark_inference - INFO - Processing 10 string columns: ['trx_channel', 'trx_type', 'mbar_registered_channel', 'mbar_a_c_status', 'mbar_a_c_level', 'mbar_account_type_name', 'user_most_used_channel_7d', 'user_last_used_channel', 'user_most_used_type_7d', 'user_last_used_type']
2025-11-05 11:47:52,471 - spark_inference - INFO - 🔧 Fitting string indexers...
2025-11-05 11:47:52,471 - spark_inference - INFO - 🔧 Fitting string indexers...
2025-11-05 11:47:57,942 - spark_inference - INFO - ✅ String indexing completed and checkpointed
2025-11-05 11:47:57,943 - spark_i

✅ String indexing and one-hot encoding completed


In [ ]:
# EXACTLY MATCHING TRAINING - Remove first feature  
modeling_features = modeling_features[1:]

logger.info(f"🧮 Final modeling features (after removing first): {len(modeling_features)} features")
print(f"Features for modeling: {len(modeling_features)}")

2025-11-05 11:47:58,834 - spark_inference - INFO - 🧮 Features for inference: 77 features


Features for modeling: 77


In [ ]:
# Create feature vectors using selected features - EXACTLY MATCHING TRAINING
logger.info("? Creating feature vectors for Spark MLlib...")

# Create feature vector using VectorAssembler
assembler = VectorAssembler(
    inputCols=modeling_features,
    outputCol="raw_features",
    handleInvalid="skip"  # Skip rows with invalid values
)

# Apply the assembler
logger.info(f"   • Assembling {len(modeling_features)} features into vector...")
df_assembled = assembler.transform(df_clean)
logger.info("✅ Feature vector created")

print("✅ Feature vectors created")

2025-11-05 11:47:58,841 - spark_inference - INFO - 📏 Creating feature vectors...


✅ Feature vectors created


In [ ]:
# Scale features using StandardScaler - EXACTLY MATCHING TRAINING
logger.info("? Scaling features...")
scaler = StandardScaler(
    inputCol="raw_features",
    outputCol="features",
    withStd=True,  # Scale to unit variance
    withMean=True  # Center the data
)

# Fit and transform the scaler
logger.info("   • Fitting scaler on inference data...")
scaler_model = scaler.fit(df_assembled)
df_scaled = scaler_model.transform(df_assembled)
logger.info("✅ Features scaled successfully")

# Prepare final dataset for inference (keeping additional columns for results)
inference_df = df_scaled.select("features", col(TARGET_COLUMN).alias("label"), "trans_id", "ac_from", "ac_to")

logger.info("✅ Final dataset prepared:")
logger.info(f"   • Features: Vector of {len(modeling_features)} elements")
logger.info(f"   • Label: Binary (0=Legitimate, 1=Fraud)")
logger.info(f"   • Records: {inference_df.count():,}")

# Show sample of the prepared data
logger.info("📋 Sample of Prepared Data:")
inference_df.show(3, truncate=False)

print("✅ Feature scaling completed")

2025-11-05 11:47:58,906 - spark_inference - INFO - 📊 Resource usage monitoring...
2025-11-05 11:47:58,908 - spark_inference - INFO - 📈 DataFrame info:
2025-11-05 11:47:58,908 - spark_inference - INFO - 📈 DataFrame info:
2025-11-05 11:47:58,920 - spark_inference - INFO -    • Partitions: 9
2025-11-05 11:47:58,921 - spark_inference - INFO -    • Columns: 102
2025-11-05 11:47:58,922 - spark_inference - INFO -    • Cached: False
2025-11-05 11:47:58,922 - spark_inference - INFO - 🧮 Feature statistics:
2025-11-05 11:47:58,923 - spark_inference - INFO -    • Total modeling features: 77
2025-11-05 11:47:58,923 - spark_inference - INFO -    • String columns processed: 10
2025-11-05 11:47:58,924 - spark_inference - INFO -    • Numeric columns: 67
2025-11-05 11:47:58,926 - spark_inference - INFO - 💾 Storage info: Serialized 1x Replicated
2025-11-05 11:47:58,920 - spark_inference - INFO -    • Partitions: 9
2025-11-05 11:47:58,921 - spark_inference - INFO -    • Columns: 102
2025-11-05 11:47:58,92

✅ Ready for feature assembly with 77 features


In [ ]:
# Scale features
scaler = StandardScaler(
    inputCol="raw_features",
    outputCol="features",
    withStd=True,
    withMean=True
)

scaler_model = scaler.fit(df_assembled)
df_scaled = scaler_model.transform(df_assembled)

logger.info("✅ Feature scaling completed")
print("✅ Feature scaling completed")

2025-11-05 11:38:53,173 - spark_inference - INFO - ✅ Feature scaling completed


✅ Feature scaling completed


## Model Inference

Run the trained model on the preprocessed data to generate predictions.

In [ ]:
predictions = lr_model.transform(df_scaled)

In [ ]:
predictions.show()

25/11/05 11:40:23 WARN TaskSetManager: Lost task 0.0 in stage 75.0 (TID 244) (10.205.161.118 executor 1): org.apache.spark.SparkException: [FAILED_EXECUTE_UDF] User defined function (`ProbabilisticClassificationModel$$Lambda/0x00007ee408f27708`: (struct<type:tinyint,size:int,indices:array<int>,values:array<double>>) => struct<type:tinyint,size:int,indices:array<int>,values:array<double>>) failed due to: java.lang.IllegalArgumentException: requirement failed: BLAS.dot(x: Vector, y:Vector) was given Vectors with non-matching sizes: x.size = 206, y.size = 290. SQLSTATE: 39000
	at org.apache.spark.sql.errors.QueryExecutionErrors$.failedExecuteUserDefinedFunctionError(QueryExecutionErrors.scala:195)
	at org.apache.spark.sql.errors.QueryExecutionErrors.failedExecuteUserDefinedFunctionError(QueryExecutionErrors.scala)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$SpecificUnsafeProjection.writeFields_0_29$(Unknown Source)
	at org.apache.spark.sql.catalyst.expressions.GeneratedCl

Py4JJavaError: An error occurred while calling o2097.showString.
: org.apache.spark.SparkException: [FAILED_EXECUTE_UDF] User defined function (`ProbabilisticClassificationModel$$Lambda/0x00007ee408f27708`: (struct<type:tinyint,size:int,indices:array<int>,values:array<double>>) => struct<type:tinyint,size:int,indices:array<int>,values:array<double>>) failed due to: java.lang.IllegalArgumentException: requirement failed: BLAS.dot(x: Vector, y:Vector) was given Vectors with non-matching sizes: x.size = 206, y.size = 290. SQLSTATE: 39000
	at org.apache.spark.sql.errors.QueryExecutionErrors$.failedExecuteUserDefinedFunctionError(QueryExecutionErrors.scala:195)
	at org.apache.spark.sql.errors.QueryExecutionErrors.failedExecuteUserDefinedFunctionError(QueryExecutionErrors.scala)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$SpecificUnsafeProjection.writeFields_0_29$(Unknown Source)
	at org.apache.spark.sql.catalyst.expressions.GeneratedClass$SpecificUnsafeProjection.apply(Unkn

In [ ]:
# Run inference using the trained model
logger.info("🔮 Running inference on prepared data...")

start_time = datetime.now()

# Make predictions using the exact same preprocessing as training
predictions = lr_model.transform(inference_df)

# Select relevant columns including transaction identifiers
results = predictions.select(
    'trans_id',
    'ac_from', 
    'ac_to',
    col('label').alias('actual_fraud_flag'),
    col('prediction').alias('predicted_fraud_flag'),
    col('probability').alias('fraud_probability')
)

results.cache()

inference_time = (datetime.now() - start_time).total_seconds()
result_count = results.count()

logger.info(f"✅ Inference completed in {inference_time:.2f} seconds")
logger.info(f"   • Records scored: {result_count:,}")

print(f"✅ Inference completed in {inference_time:.2f} seconds")
print(f"Records scored: {result_count:,}")

2025-11-05 11:39:27,571 - spark_inference - INFO - 🔮 Running inference on test data...


DataFrame[trans_id: string, ac_from: string, ac_to: string, actual_fraud_flag: int, predicted_fraud_flag: double, fraud_probability: vector]

## Results Analysis

Analyze the inference results and compare with actual labels.

In [ ]:
# Display predictions distribution
logger.info("📊 Inference Results Summary:")

pred_dist = results.groupBy('predicted_fraud_flag').count().collect()
for row in pred_dist:
    label = "🚨 Fraud" if row['predicted_fraud_flag'] == 1.0 else "✅ Non-Fraud"
    logger.info(f"   • {label}: {row['count']:,} records")
    print(f"{label}: {row['count']:,} records")

print("\nPrediction Distribution:")
results.groupBy('predicted_fraud_flag').count().show()

In [ ]:
# Show sample predictions
logger.info("\n📋 Sample Predictions:")
print("Sample Predictions:")
results.show(10, truncate=False)

In [ ]:
# Calculate accuracy against actual fraud_flag
logger.info("📊 Comparing predictions with actual labels...")

comparison = results.withColumn(
    'correct',
    when(
        (col('predicted_fraud_flag') == 1) & (col('actual_fraud_flag') == 1), 'TP'
    ).when(
        (col('predicted_fraud_flag') == 1) & (col('actual_fraud_flag') == 0), 'FP'
    ).when(
        (col('predicted_fraud_flag') == 0) & (col('actual_fraud_flag') == 1), 'FN'
    ).otherwise('TN')
)

confusion = comparison.groupBy('correct').count().collect()

logger.info("\n📋 Confusion Matrix:")
print("\nConfusion Matrix:")
for row in confusion:
    logger.info(f"   • {row['correct']}: {row['count']:,}")
    print(f"{row['correct']}: {row['count']:,}")

In [ ]:
# Calculate metrics
tp = next((row['count'] for row in confusion if row['correct'] == 'TP'), 0)
fp = next((row['count'] for row in confusion if row['correct'] == 'FP'), 0)
fn = next((row['count'] for row in confusion if row['correct'] == 'FN'), 0)
tn = next((row['count'] for row in confusion if row['correct'] == 'TN'), 0)

accuracy = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else 0
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

logger.info(f"\n🎯 Performance Metrics:")
logger.info(f"   • Accuracy: {accuracy:.4f}")
logger.info(f"   • Precision: {precision:.4f}")
logger.info(f"   • Recall: {recall:.4f}")
logger.info(f"   • F1-Score: {f1:.4f}")

print(f"\n🎯 Performance Metrics:")
print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-Score: {f1:.4f}")

## Save Results

Save the inference results to Parquet format for further analysis.

In [ ]:
# Save results to Parquet
logger.info("\n💾 Saving inference results to Parquet...")
output_path = f"/root/research-dir/dev/jazzcash-fraud-detection/results/fraud_predictions_{datetime.now().strftime('%Y%m%d_%H%M%S')}"
os.makedirs(os.path.dirname(output_path), exist_ok=True)
results.write.mode("overwrite").parquet(output_path)

logger.info(f"✅ Results saved to Parquet:")
logger.info(f"   • Path: {output_path}")

print(f"✅ Results saved to: {output_path}")

In [ ]:
# Clean up
results.unpersist()
logger.info("\n🧹 Cache cleaned - Spark session ready")
print("🧹 Cache cleaned - Spark session ready")

## Summary

This notebook has successfully performed fraud detection inference using the trained Logistic Regression model. Key accomplishments:

1. ✅ Loaded inference data from ClickHouse
2. ✅ Applied feature preprocessing and scaling
3. ✅ Generated fraud predictions
4. ✅ Evaluated model performance
5. ✅ Saved results to Parquet format

The results can be used for further analysis or integrated into production fraud detection systems.